In [68]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt
import torch.optim as optim

### Seed + device

In [69]:
seed = 200

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
elif torch.backends.mps.is_available():
    torch.mps.manual_seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

### Sprawdzenie średniej i odchylenia dla każdego kanału

In [70]:
transform = transforms.Compose([transforms.ToTensor()])
dataset = torchvision.datasets.ImageFolder("train/", transform=transform)

map_class_to_idx = dataset.class_to_idx

all_samples = torch.stack([sample[0] for sample in dataset])
print(f"Data norm per channel: mean={all_samples.mean(axis=(0,2,3))} | std={all_samples.std(axis=(0,2,3))}")

Data norm per channel: mean=tensor([0.5204, 0.4950, 0.4381]) | std=tensor([0.2841, 0.2770, 0.2974])


### Transformacje na zbiorach

In [71]:
train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomCrop(64, 4),
        transforms.RandomRotation(16),
        transforms.ToTensor(),
        transforms.Normalize((0.5204, 0.4950, 0.4381), (0.2841, 0.2770, 0.2974))])

val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5204, 0.4950, 0.4381), (0.2841, 0.2770, 0.2974))])

train_dataset = torchvision.datasets.ImageFolder("train/", transform=train_transform)
val_dataset = torchvision.datasets.ImageFolder("train/", transform=val_transform)

dataset_size = len(train_dataset)
train_size = int(0.85 * dataset_size)
val_size = dataset_size - train_size

indices = torch.randperm(dataset_size)

train_subset = torch.utils.data.Subset(train_dataset, indices[:train_size])
val_subset = torch.utils.data.Subset(val_dataset, indices[train_size:])

print(f"Train: {len(train_subset)}")
print(f"Val: {len(val_subset)}")


Train: 74809
Val: 13202


### Dataloadery do treningu i walidacji

In [72]:
train_loader = torch.utils.data.DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_subset, batch_size=64, shuffle=True)

In [ ]:
class Net(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.conv_layer1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.conv_layer2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.conv_layer3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.dense_layer = nn.Sequential(
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, n_classes)
        )
    
    def forward(self, x):
        x = self.conv_layer1(x)
        x = self.conv_layer2(x)
        x = self.conv_layer3(x)
        x = torch.flatten(x, 1)
        x = self.dense_layer(x)
        return x

In [79]:
model = Net(len(map_class_to_idx)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

In [80]:
def calc_accuracy(predictions: np.ndarray, targets: np.ndarray, n_classes=50):
    assert len(predictions) == len(targets)
    accuracies = []
    for i in range(n_classes):
        accuracies.append((predictions[targets == i] == i).sum() / (targets == i).sum())
    return np.mean(accuracies)